<div style="background:linear-gradient(160deg,#0f1117,#1a1d2e);border:1px solid #2a2d45;border-radius:16px;padding:48px 40px;margin:8px 0 32px;">
<div style="font-family:monospace;font-size:11px;color:#7c8cff;letter-spacing:.15em;text-transform:uppercase;margin-bottom:12px;">Chapters 3 · 4 · 5 · 6</div>
<h1 style="font-size:34px;font-weight:300;color:#e8e9f0;line-height:1.2;margin:0 0 10px;">Build a <em style="color:#a78bfa;">GPT-2</em> From Scratch</h1>
<div style="font-size:14px;color:#9fa3b8;margin-bottom:24px;">Build a Large Language Model From Scratch · Sebastian Raschka · Summer School Demo</div>
<div style="display:flex;gap:10px;flex-wrap:wrap;">
<span style="background:#7c8cff18;border:1px solid #7c8cff30;color:#7c8cff;font-family:monospace;font-size:11px;padding:4px 12px;border-radius:20px;">PyTorch 2.x</span>
<span style="background:#4ade8018;border:1px solid #4ade8030;color:#4ade80;font-family:monospace;font-size:11px;padding:4px 12px;border-radius:20px;">Python 3.11</span>
<span style="background:#f87171" + "18;border:1px solid #f8717130;color:#f87171;font-family:monospace;font-size:11px;padding:4px 12px;border-radius:20px;">tiktoken</span>
</div>
</div>


**Full notebook roadmap:**

- **Ch. 3** — Self-attention → Trainable weights → Causal attention → Multi-head attention
- **Ch. 4** — Layer Norm · GELU · FeedForward · Shortcut Connections · TransformerBlock · GPTModel
- **Ch. 5** — Training loop · Cross-entropy loss · Text generation
- **Ch. 6** — Load pretrained GPT-2 weights from OpenAI · Inference demo


<div style="background:linear-gradient(135deg,#4ade8018,#4ade8005);border-left:4px solid #4ade80;border-radius:0 12px 12px 0;padding:20px 28px;margin:32px 0 8px;">
<span style="font-family:monospace;font-size:11px;color:#4ade80;letter-spacing:.12em;text-transform:uppercase;">section 00</span><br>
<strong style="font-size:22px;color:#e8e9f0;">Environment Setup</strong><br>
<span style="font-size:13px;color:#9fa3b8;">Verify packages before we start</span>
</div>


In [ ]:
from importlib.metadata import version
import torch
import torch.nn as nn

print("torch version:", version("torch"))
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch version: 2.12.0+cpu
CUDA available: False
Using device: cpu


<div style="background:linear-gradient(135deg,#7c8cff18,#7c8cff05);border-left:4px solid #7c8cff;border-radius:0 12px 12px 0;padding:20px 28px;margin:32px 0 8px;">
<span style="font-family:monospace;font-size:11px;color:#7c8cff;letter-spacing:.12em;text-transform:uppercase;">section 01</span><br>
<strong style="font-size:22px;color:#e8e9f0;">Simple Self-Attention</strong><br>
<span style="font-size:13px;color:#9fa3b8;">No trainable weights — understanding the core mechanics</span>
</div>


### Setting up: 6 token embeddings in $\mathbb{R}^3$

Each row is a word turned into numbers. In a real LLM this would be $\mathbb{R}^{768}$.


In [ ]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

<div style="background:#1e2030;border-left:3px solid #fbbf24;border-radius:0 10px 10px 0;padding:16px 22px;margin:12px 0;">
<div style="font-family:monospace;font-size:14px;color:#fbbf24;text-align:center;">ω₂ⱼ = x⁽²⁾ · x⁽ʲ⁾ = Σₖ x⁽²⁾ₖ · x⁽ʲ⁾ₖ</div>
<div style="font-size:12px;color:#9fa3b8;margin-top:8px;">Dot product = unnormalized similarity score between query and each token</div>
</div>


In [ ]:
query = inputs[1]  # "journey" as query
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


<div style="background:#fbbf2412;border:1px solid #fbbf2430;border-radius:10px;padding:14px 18px;margin:10px 0;font-size:13px;color:#c8cadc;line-height:1.7;">
<strong style="color:#fbbf24;">⚠️ Note —</strong> These raw scores are <em>not</em> probabilities — they have no upper bound and do not sum to 1. Softmax converts them to a proper probability distribution.
</div>


In [ ]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


<div style="background:#1e2030;border-left:3px solid #fbbf24;border-radius:0 10px 10px 0;padding:16px 22px;margin:12px 0;">
<div style="font-family:monospace;font-size:14px;color:#fbbf24;text-align:center;">z⁽²⁾ = Σᵢ α₂ᵢ · x⁽ⁱ⁾   where Σᵢ α₂ᵢ = 1</div>
<div style="font-size:12px;color:#9fa3b8;margin-top:8px;">Context vector: a weighted average encoding 'journey in the context of the whole sentence'</div>
</div>


In [ ]:
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


### Vectorized: all context vectors at once


In [ ]:
attn_scores = inputs @ inputs.T          # [6,3]@[3,6] = [6,6]
attn_weights = torch.softmax(attn_scores, dim=-1)  # normalize each ROW
all_context_vecs = attn_weights @ inputs  # [6,6]@[6,3] = [6,3]
print(all_context_vecs)
print("Sanity check row 1:", all_context_vecs[1])  # must match context_vec_2

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
Sanity check row 1: tensor([0.4419, 0.6515, 0.5683])


<div style="background:linear-gradient(135deg,#2dd4bf18,#2dd4bf05);border-left:4px solid #2dd4bf;border-radius:0 12px 12px 0;padding:20px 28px;margin:32px 0 8px;">
<span style="font-family:monospace;font-size:11px;color:#2dd4bf;letter-spacing:.12em;text-transform:uppercase;">section 02</span><br>
<strong style="font-size:22px;color:#e8e9f0;">Self-Attention with Trainable Weights</strong><br>
<span style="font-size:13px;color:#9fa3b8;">Introducing learnable W_Q, W_K, W_V projections</span>
</div>


<div style="background:#1e2030;border-left:3px solid #fbbf24;border-radius:0 10px 10px 0;padding:16px 22px;margin:12px 0;">
<div style="font-family:monospace;font-size:14px;color:#fbbf24;text-align:center;">Attention(Q,K,V) = softmax( QKᵀ / √d_k ) · V</div>
<div style="font-size:12px;color:#9fa3b8;margin-top:8px;">Three independent projections: what to search for (Q), what to expose (K), what to contribute (V)</div>
</div>


In [ ]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))
    def forward(self, x):
        keys    = x @ self.W_key
        queries = x @ self.W_query
        values  = x @ self.W_value
        attn_scores  = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        return attn_weights @ values

torch.manual_seed(123)
d_in, d_out = 3, 2
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


<div style="background:#f472b612;border:1px solid #f472b630;border-radius:10px;padding:14px 18px;margin:10px 0;font-size:13px;color:#c8cadc;line-height:1.7;">
<strong style="color:#f472b6;">💡 Note —</strong> <code>nn.Parameter(torch.rand(...))</code> uses Uniform[0,1) — poor initialization. <code>nn.Linear</code> uses Kaiming uniform automatically, which is more stable for training.
</div>


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    def forward(self, x):
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)
        attn_scores  = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        return attn_weights @ values

torch.manual_seed(789)
sa_v2 = SelfAttention(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


<div style="background:linear-gradient(135deg,#f8717118,#f8717105);border-left:4px solid #f87171;border-radius:0 12px 12px 0;padding:20px 28px;margin:32px 0 8px;">
<span style="font-family:monospace;font-size:11px;color:#f87171;letter-spacing:.12em;text-transform:uppercase;">section 03</span><br>
<strong style="font-size:22px;color:#e8e9f0;">Causal Attention & Dropout</strong><br>
<span style="font-size:13px;color:#9fa3b8;">Masking future tokens — essential for autoregressive LLMs</span>
</div>


<div style="background:#1e2030;border-left:3px solid #fbbf24;border-radius:0 10px 10px 0;padding:16px 22px;margin:12px 0;">
<div style="font-family:monospace;font-size:14px;color:#fbbf24;text-align:center;">Mᵢⱼ = 0 if j ≤ i,   −∞ if j > i</div>
<div style="font-size:12px;color:#9fa3b8;margin-top:8px;">exp(−∞) = 0 after softmax → future positions receive exactly zero weight</div>
</div>


In [ ]:
queries = sa_v2.W_query(inputs)
keys    = sa_v2.W_key(inputs)
attn_scores  = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

context_length = attn_scores.shape[0]
mask_simple    = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [ ]:
# More efficient: fill -inf BEFORE softmax (correct approach)
mask   = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [ ]:
torch.manual_seed(123)
dropout = nn.Dropout(0.5)
example = torch.ones(6, 6)
print(dropout(example))  # surviving values scaled by 1/(1-0.5)=2.0

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out    = d_out
        self.W_query  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key    = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout  = nn.Dropout(dropout)
        # register_buffer: constant, no gradient, moves with .to(device)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)
        attn_scores = queries @ keys.transpose(1, 2)  # NOT .T for 3D tensors!
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        return attn_weights @ values

batch = torch.stack((inputs, inputs), dim=0)  # [2, 6, 3]
torch.manual_seed(123)
ca = CausalAttention(d_in=3, d_out=2, context_length=6, dropout=0.0)
context_vecs = ca(batch)
print(context_vecs.shape)  # [2, 6, 2]

torch.Size([2, 6, 2])


<div style="background:linear-gradient(135deg,#4ade8018,#4ade8005);border-left:4px solid #4ade80;border-radius:0 12px 12px 0;padding:20px 28px;margin:32px 0 8px;">
<span style="font-family:monospace;font-size:11px;color:#4ade80;letter-spacing:.12em;text-transform:uppercase;">section 04</span><br>
<strong style="font-size:22px;color:#e8e9f0;">Multi-Head Attention</strong><br>
<span style="font-size:13px;color:#9fa3b8;">Parallel attention heads — the core of every modern Transformer</span>
</div>


<div style="background:#1e2030;border-left:3px solid #fbbf24;border-radius:0 10px 10px 0;padding:16px 22px;margin:12px 0;">
<div style="font-family:monospace;font-size:14px;color:#fbbf24;text-align:center;">MultiHead(X) = Concat(head₁,...,headₕ)·W_O    head_i = Attention(XW_Qi, XW_Ki, XW_Vi)</div>
<div style="font-size:12px;color:#9fa3b8;margin-top:8px;">Each head learns a different subspace — emerges from training, not hardcoded</div>
</div>


In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    """Simple stacking approach — correct but less efficient"""
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )
    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

torch.manual_seed(123)
mha_wrap = MultiHeadAttentionWrapper(d_in=3, d_out=2, context_length=6,
                                      dropout=0.0, num_heads=2)
print(mha_wrap(batch).shape)  # [2, 6, 4]

torch.Size([2, 6, 4])


<div style="display:flex;align-items:center;flex-wrap:wrap;gap:6px;padding:12px 4px;">
<div style="background:#252840;border:1px solid #3a3d5c;border-radius:8px;padding:8px 16px;text-align:center;"><div style="font-size:10px;color:#666880;margin-bottom:4px;">input x</div><div style="font-family:monospace;font-size:12px;color:#2dd4bf;font-weight:500;">[b, T, d_in]</div></div><span style="color:#666880;font-size:16px;padding:0 6px;">→</span><div style="background:#252840;border:1px solid #3a3d5c;border-radius:8px;padding:8px 16px;text-align:center;"><div style="font-size:10px;color:#666880;margin-bottom:4px;">W_key(x)</div><div style="font-family:monospace;font-size:12px;color:#2dd4bf;font-weight:500;">[b, T, d_out]</div></div><span style="color:#666880;font-size:16px;padding:0 6px;">→</span><div style="background:#252840;border:1px solid #3a3d5c;border-radius:8px;padding:8px 16px;text-align:center;"><div style="font-size:10px;color:#666880;margin-bottom:4px;">.view(b,T,h,hd)</div><div style="font-family:monospace;font-size:12px;color:#2dd4bf;font-weight:500;">[b, T, h, head_dim]</div></div><span style="color:#666880;font-size:16px;padding:0 6px;">→</span><div style="background:#252840;border:1px solid #3a3d5c;border-radius:8px;padding:8px 16px;text-align:center;"><div style="font-size:10px;color:#666880;margin-bottom:4px;">.transpose(1,2)</div><div style="font-family:monospace;font-size:12px;color:#2dd4bf;font-weight:500;">[b, h, T, head_dim]</div></div><span style="color:#666880;font-size:16px;padding:0 6px;">→</span><div style="background:#252840;border:1px solid #3a3d5c;border-radius:8px;padding:8px 16px;text-align:center;"><div style="font-size:10px;color:#666880;margin-bottom:4px;">scores</div><div style="font-family:monospace;font-size:12px;color:#2dd4bf;font-weight:500;">[b, h, T, T]</div></div><span style="color:#666880;font-size:16px;padding:0 6px;">→</span><div style="background:#252840;border:1px solid #3a3d5c;border-radius:8px;padding:8px 16px;text-align:center;"><div style="font-size:10px;color:#666880;margin-bottom:4px;">output</div><div style="font-family:monospace;font-size:12px;color:#2dd4bf;font-weight:500;">[b, T, d_out]</div></div>
</div>


In [ ]:
class MultiHeadAttention(nn.Module):
    """Efficient weight-split approach — one large GEMM instead of h small ones"""
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_out     = d_out
        self.num_heads = num_heads
        self.head_dim  = d_out // num_heads
        self.W_query   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key     = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj  = nn.Linear(d_out, d_out)  # W_O
        self.dropout   = nn.Dropout(dropout)
        self.register_buffer("mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)
        # Reshape: split d_out into (num_heads, head_dim)
        keys    = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1,2)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1,2)
        values  = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1,2)
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        # Merge heads: .contiguous() required before .view() after transpose
        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

torch.manual_seed(123)
mha = MultiHeadAttention(d_in=3, d_out=2, context_length=6, dropout=0.0, num_heads=2)
print(mha(batch).shape)  # [2, 6, 2]

torch.Size([2, 6, 2])


<div style="background:linear-gradient(135deg,#a78bfa18,#a78bfa05);border-left:4px solid #a78bfa;border-radius:0 12px 12px 0;padding:20px 28px;margin:32px 0 8px;">
<span style="font-family:monospace;font-size:11px;color:#a78bfa;letter-spacing:.12em;text-transform:uppercase;">section 05</span><br>
<strong style="font-size:22px;color:#e8e9f0;">Chapter 4 — GPT Architecture</strong><br>
<span style="font-size:13px;color:#9fa3b8;">Layer Norm · GELU · FeedForward · Shortcut Connections · TransformerBlock · GPTModel</span>
</div>


### GPT-2 (124M) Configuration

The smallest GPT-2 model uses these hyperparameters:


In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size (BPE tokenizer)
    "context_length": 1024, # Max sequence length
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of transformer blocks
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # No QKV bias (modern standard)
}

### 4.1 — DummyGPTModel (placeholder to see the big picture)


In [ ]:
class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb   = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb   = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb  = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head   = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)

class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg): super().__init__()
    def forward(self, x): return x  # pass-through placeholder

class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5): super().__init__()
    def forward(self, x): return x  # pass-through placeholder

import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
batch_txt = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch_txt.append(torch.tensor(tokenizer.encode(txt1)))
batch_txt.append(torch.tensor(tokenizer.encode(txt2)))
batch_txt = torch.stack(batch_txt, dim=0)

torch.manual_seed(123)
model_dummy = DummyGPTModel(GPT_CONFIG_124M)
logits = model_dummy(batch_txt)
print("Output shape:", logits.shape)  # [2, 4, 50257]

Output shape: torch.Size([2, 4, 50257])


<div style="background:linear-gradient(135deg,#fbbf2418,#fbbf2405);border-left:4px solid #fbbf24;border-radius:0 12px 12px 0;padding:20px 28px;margin:32px 0 8px;">
<span style="font-family:monospace;font-size:11px;color:#fbbf24;letter-spacing:.12em;text-transform:uppercase;">section 06</span><br>
<strong style="font-size:22px;color:#e8e9f0;">4.2 — Layer Normalization</strong><br>
<span style="font-size:13px;color:#9fa3b8;">Stabilize training by normalizing activations to mean=0, variance=1</span>
</div>


<div style="background:#1e2030;border-left:3px solid #fbbf24;border-radius:0 10px 10px 0;padding:16px 22px;margin:12px 0;">
<div style="font-family:monospace;font-size:14px;color:#fbbf24;text-align:center;">x̂ = (x − μ) / √(σ² + ε)   then   out = scale · x̂ + shift</div>
<div style="font-size:12px;color:#9fa3b8;margin-top:8px;">scale and shift are learnable parameters — the model adjusts them during training</div>
</div>


In [ ]:
torch.manual_seed(123)
batch_example = torch.randn(2, 5)
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out_ex = layer(batch_example)
print(out_ex)

mean = out_ex.mean(dim=-1, keepdim=True)
var  = out_ex.var(dim=-1, keepdim=True)
print("Mean:\n", mean)
print("Variance:\n", var)

tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)
Mean:
 tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)


In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps   = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
        mean   = x.mean(dim=-1, keepdim=True)
        var    = x.var(dim=-1, keepdim=True, unbiased=False)  # biased for GPT-2 compat
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)
print("Mean:\n",     out_ln.mean(dim=-1, keepdim=True))
print("Variance:\n", out_ln.var(dim=-1, unbiased=False, keepdim=True))

Mean:
 tensor([[-2.9802e-08],
        [ 0.0000e+00]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


<div style="background:linear-gradient(135deg,#2dd4bf18,#2dd4bf05);border-left:4px solid #2dd4bf;border-radius:0 12px 12px 0;padding:20px 28px;margin:32px 0 8px;">
<span style="font-family:monospace;font-size:11px;color:#2dd4bf;letter-spacing:.12em;text-transform:uppercase;">section 07</span><br>
<strong style="font-size:22px;color:#e8e9f0;">4.3 — GELU Activation & FeedForward Module</strong><br>
<span style="font-size:13px;color:#9fa3b8;">Smooth non-linearity preferred over ReLU in modern LLMs</span>
</div>


<div style="background:#1e2030;border-left:3px solid #fbbf24;border-radius:0 10px 10px 0;padding:16px 22px;margin:12px 0;">
<div style="font-family:monospace;font-size:14px;color:#fbbf24;text-align:center;">GELU(x) = x · Φ(x)   where Φ is the Gaussian CDF</div>
<div style="font-size:12px;color:#9fa3b8;margin-top:8px;">Differentiable everywhere unlike ReLU — allows smooth gradient flow even for negative values</div>
</div>


In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

# Quick visual comparison: GELU vs ReLU
import matplotlib.pyplot as plt
x = torch.linspace(-3, 3, 100)
gelu, relu = GELU(), nn.ReLU()
plt.figure(figsize=(8,3))
plt.plot(x.numpy(), gelu(x).detach().numpy(), label="GELU", linewidth=2, color="#7c8cff")
plt.plot(x.numpy(), relu(x).detach().numpy(), label="ReLU", linewidth=2, color="#f87171", linestyle="--")
plt.title("GELU vs ReLU activation function")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("gelu_vs_relu.png", dpi=100)
plt.show()
print("GELU is smooth even at x=0 — better gradient properties")

GELU is smooth even at x=0 — better gradient properties


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),  # expand 4x
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),  # contract back
        )
    def forward(self, x):
        return self.layers(x)

ffn = FeedForward(GPT_CONFIG_124M)
x_test = torch.rand(2, 3, 768)
print("FeedForward input shape:", x_test.shape)
print("FeedForward output shape:", ffn(x_test).shape)  # same shape!

FeedForward input shape: torch.Size([2, 3, 768])
FeedForward output shape: torch.Size([2, 3, 768])


<div style="background:#7c8cff12;border:1px solid #7c8cff30;border-radius:10px;padding:14px 18px;margin:10px 0;font-size:13px;color:#c8cadc;line-height:1.7;">
<strong style="color:#7c8cff;">💡 Note —</strong> The FeedForward expands to <code>4 × emb_dim = 3072</code> then contracts back. This "expand then squeeze" creates a bottleneck where the model can compute complex non-linear interactions. The 4× factor is an empirical choice from the original Transformer paper.
</div>


<div style="background:linear-gradient(135deg,#f472b618,#f472b605);border-left:4px solid #f472b6;border-radius:0 12px 12px 0;padding:20px 28px;margin:32px 0 8px;">
<span style="font-family:monospace;font-size:11px;color:#f472b6;letter-spacing:.12em;text-transform:uppercase;">section 08</span><br>
<strong style="font-size:22px;color:#e8e9f0;">4.4 — Shortcut Connections</strong><br>
<span style="font-size:13px;color:#9fa3b8;">Residual paths that prevent vanishing gradients in deep networks</span>
</div>


<div style="background:#1e2030;border-left:3px solid #fbbf24;border-radius:0 10px 10px 0;padding:16px 22px;margin:12px 0;">
<div style="font-family:monospace;font-size:14px;color:#fbbf24;text-align:center;">x_out = LayerNorm(x) → SubLayer(x) → Dropout → + x_in</div>
<div style="font-size:12px;color:#9fa3b8;margin-top:8px;">The skip connection adds the input directly to the output — gradient flows through both paths</div>
</div>


In [ ]:
import torch
import torch.nn as nn


class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut

        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), nn.GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), nn.GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), nn.GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), nn.GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), nn.GELU()),
        ])

    def forward(self, x):
        for layer in self.layers:
            layer_output = layer(x)

            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output  # residual / shortcut connection
            else:
                x = layer_output

        return x


def print_gradients(model, x):
    model.zero_grad(set_to_none=True)

    output = model(x)
    target = torch.tensor([[0.]], dtype=output.dtype, device=output.device)

    loss = nn.MSELoss()(output, target)
    loss.backward()

    for name, param in model.named_parameters():
        if "weight" in name:
            grad_mean = param.grad.abs().mean().item()
            print(f"{name}: gradient mean = {grad_mean:.6f}")


layer_sizes = [3, 3, 3, 3, 3, 1]
sample_input = torch.tensor([[1., 0., -1.]])

print("=== WITHOUT shortcut connections ===")
torch.manual_seed(123)
model_no_sc = ExampleDeepNeuralNetwork(layer_sizes, use_shortcut=False)
print_gradients(model_no_sc, sample_input)

print("\n=== WITH shortcut connections ===")
torch.manual_seed(123)
model_sc = ExampleDeepNeuralNetwork(layer_sizes, use_shortcut=True)
print_gradients(model_sc, sample_input)

=== WITHOUT shortcut connections ===
layers.0.0.weight: gradient mean = 0.000202
layers.1.0.weight: gradient mean = 0.000120
layers.2.0.weight: gradient mean = 0.000715
layers.3.0.weight: gradient mean = 0.001399
layers.4.0.weight: gradient mean = 0.005050

=== WITH shortcut connections ===
layers.0.0.weight: gradient mean = 0.221868
layers.1.0.weight: gradient mean = 0.207093
layers.2.0.weight: gradient mean = 0.329239
layers.3.0.weight: gradient mean = 0.266777
layers.4.0.weight: gradient mean = 1.326806


<div style="background:#7c8cff12;border:1px solid #7c8cff30;border-radius:10px;padding:14px 18px;margin:10px 0;font-size:13px;color:#c8cadc;line-height:1.7;">
<strong style="color:#7c8cff;">💡 Note —</strong> Without shortcuts, layer 0 gradient (0.0002) is <strong>25× smaller</strong> than layer 4 (0.005) — the vanishing gradient problem. With shortcuts, all layers maintain healthy gradients (~0.2–1.3). This is why every modern deep network uses residual connections.
</div>


<div style="background:linear-gradient(135deg,#7c8cff18,#7c8cff05);border-left:4px solid #7c8cff;border-radius:0 12px 12px 0;padding:20px 28px;margin:32px 0 8px;">
<span style="font-family:monospace;font-size:11px;color:#7c8cff;letter-spacing:.12em;text-transform:uppercase;">section 09</span><br>
<strong style="font-size:22px;color:#e8e9f0;">4.5 — TransformerBlock</strong><br>
<span style="font-size:13px;color:#9fa3b8;">Combining everything: MHA + LayerNorm + FeedForward + Shortcut + Dropout</span>
</div>


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff         = FeedForward(cfg)
        self.norm1      = LayerNorm(cfg["emb_dim"])
        self.norm2      = LayerNorm(cfg["emb_dim"])
        self.drop_resid = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # --- Attention sub-block with Pre-LayerNorm ---
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_resid(x)
        x = x + shortcut         # shortcut connection 1
        # --- FeedForward sub-block with Pre-LayerNorm ---
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_resid(x)
        x = x + shortcut         # shortcut connection 2
        return x

torch.manual_seed(123)
x = torch.rand(2, 4, 768)
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)
print("Input shape: ", x.shape)
print("Output shape:", output.shape)  # same! shape-preserving

Input shape:  torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])


<div style="background:#7c8cff12;border:1px solid #7c8cff30;border-radius:10px;padding:14px 18px;margin:10px 0;font-size:13px;color:#c8cadc;line-height:1.7;">
<strong style="color:#7c8cff;">💡 Note —</strong> The transformer block is <strong>shape-preserving</strong>: input and output are always <code>[b, T, emb_dim]</code>. This is by design — it allows stacking 12 (or 48) identical blocks without dimension changes.
</div>


<div style="background:linear-gradient(135deg,#4ade8018,#4ade8005);border-left:4px solid #4ade80;border-radius:0 12px 12px 0;padding:20px 28px;margin:32px 0 8px;">
<span style="font-family:monospace;font-size:11px;color:#4ade80;letter-spacing:.12em;text-transform:uppercase;">section 10</span><br>
<strong style="font-size:22px;color:#e8e9f0;">4.6 — GPTModel (Full Architecture)</strong><br>
<span style="font-size:13px;color:#9fa3b8;">Stack 12 TransformerBlocks with token + positional embeddings</span>
</div>


In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb    = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb    = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb   = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head   = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)                                 # [b, T, emb]
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))  # [T, emb]
        x = tok_embeds + pos_embeds                                       # broadcast
        x = self.drop_emb(x)
        x = self.trf_blocks(x)                                           # 12 blocks
        x = self.final_norm(x)
        logits = self.out_head(x)                                        # [b, T, vocab_size]
        return logits

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
out = model(batch_txt)
print("Input batch:\n", batch_txt)
print("\nOutput shape:", out.shape)  # [2, 4, 50257]

Input batch:
 tensor([[ 6109,  3626,  6100,   345],
        [ 6109,  1110,  6622,   257]])

Output shape: torch.Size([2, 4, 50257])


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# With weight tying (token embedding shared with output layer)
total_params_gpt2 = total_params - sum(p.numel() for p in model.out_head.parameters())
print(f"Parameters with weight tying (GPT-2 original): {total_params_gpt2:,}")

total_size_bytes = total_params * 4
total_size_mb    = total_size_bytes / (1024 * 1024)
print(f"Memory footprint (fp32): {total_size_mb:.2f} MB")

Total number of parameters: 163,009,536
Parameters with weight tying (GPT-2 original): 124,412,160
Memory footprint (fp32): 621.83 MB


<div style="background:linear-gradient(135deg,#f8717118,#f8717105);border-left:4px solid #f87171;border-radius:0 12px 12px 0;padding:20px 28px;margin:32px 0 8px;">
<span style="font-family:monospace;font-size:11px;color:#f87171;letter-spacing:.12em;text-transform:uppercase;">section 11</span><br>
<strong style="font-size:22px;color:#e8e9f0;">4.7 — Text Generation (Greedy Decoding)</strong><br>
<span style="font-size:13px;color:#9fa3b8;">Converting model logits → next token, one step at a time</span>
</div>


<div style="background:#1e2030;border-left:3px solid #fbbf24;border-radius:0 10px 10px 0;padding:16px 22px;margin:12px 0;">
<div style="font-family:monospace;font-size:14px;color:#fbbf24;text-align:center;">idx_next = argmax( softmax( model(idx)[:, -1, :] ) )</div>
<div style="font-size:12px;color:#9fa3b8;margin-top:8px;">Greedy decoding: always pick the most probable next token — simple but can be repetitive</div>
</div>


In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond  = idx[:, -context_size:]         # crop to context window
        with torch.no_grad():
            logits = model(idx_cond)
        logits    = logits[:, -1, :]               # last token logits
        probas    = torch.softmax(logits, dim=-1)
        idx_next  = torch.argmax(probas, dim=-1, keepdim=True)
        idx       = torch.cat((idx, idx_next), dim=1)
    return idx

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    return torch.tensor(encoded).unsqueeze(0)  # add batch dimension

def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

model.eval()
start_context = "Hello, I am"
encoded_tensor = text_to_token_ids(start_context, tokenizer)
out_ids = generate_text_simple(
    model=model, idx=encoded_tensor,
    max_new_tokens=6, context_size=GPT_CONFIG_124M["context_length"]
)
print("Generated (untrained model):")
print(token_ids_to_text(out_ids, tokenizer))

Generated (untrained model):
Hello, I am Featureiman Byeswickattribute argue logger


<div style="background:#fbbf2412;border:1px solid #fbbf2430;border-radius:10px;padding:14px 18px;margin:10px 0;font-size:13px;color:#c8cadc;line-height:1.7;">
<strong style="color:#fbbf24;">⚠️ Note —</strong> The model generates <strong>gibberish</strong> — it has not been trained yet! Weights are random. We need either (a) training from scratch or (b) loading pretrained weights.
</div>


<div style="background:linear-gradient(135deg,#fbbf2418,#fbbf2405);border-left:4px solid #fbbf24;border-radius:0 12px 12px 0;padding:20px 28px;margin:32px 0 8px;">
<span style="font-family:monospace;font-size:11px;color:#fbbf24;letter-spacing:.12em;text-transform:uppercase;">section 12</span><br>
<strong style="font-size:22px;color:#e8e9f0;">Chapter 5 — Pretraining GPT-2 from Scratch</strong><br>
<span style="font-size:13px;color:#9fa3b8;">Cross-entropy loss, AdamW optimizer, training loop</span>
</div>


### Why training from scratch is not practical for us today

GPT-2 (124M parameters) on a small dataset like *The Verdict* (5,145 tokens) takes **minutes**.
But real pretraining of GPT-2 on 40 GB of text takes **weeks on 8× A100 GPUs** (~$50,000 in cloud cost).

We show the code structure for completeness — then we load OpenAI's pretrained weights.


In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch  = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss   = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()
    )
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

print("calc_loss_batch and calc_loss_loader defined ✓")

calc_loss_batch and calc_loss_loader defined ✓


In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss   = calc_loss_loader(val_loader,   model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    print(token_ids_to_text(token_ids, tokenizer).replace("\n", " "))
    model.train()

def train_model_simple(model, train_loader, val_loader, optimizer, device,
                        num_epochs, eval_freq, eval_iter, start_context):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1
    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")
        generate_and_print_sample(model, train_loader.dataset.tokenizer,
                                  device, start_context)
    return train_losses, val_losses, track_tokens_seen

print("Training functions defined ✓")
print("NOTE: We skip actual training here and load pretrained GPT-2 weights instead.")

Training functions defined ✓
NOTE: We skip actual training here and load pretrained GPT-2 weights instead.


<div style="background:linear-gradient(135deg,#a78bfa18,#a78bfa05);border-left:4px solid #a78bfa;border-radius:0 12px 12px 0;padding:20px 28px;margin:32px 0 8px;">
<span style="font-family:monospace;font-size:11px;color:#a78bfa;letter-spacing:.12em;text-transform:uppercase;">section 13</span><br>
<strong style="font-size:22px;color:#e8e9f0;">Chapter 6 — Loading Pretrained GPT-2 Weights</strong><br>
<span style="font-size:13px;color:#9fa3b8;">Skip training — use OpenAI's pretrained weights for instant coherent text</span>
</div>


### Why we load pretrained weights instead of training

| | Training from scratch | Loading pretrained GPT-2 |
|---|---|---|
| Dataset | 40 GB of text | Already done by OpenAI |
| Compute | 8× A100, weeks | Download 500 MB file |
| Cost | ~$50,000 | Free |
| Result | Potentially good | Production-quality GPT-2 |

Training LLMs from scratch is reserved for organizations with massive compute budgets. For demos, research, and fine-tuning — we always start from pretrained weights.


In [ ]:
import urllib.request

# Download the GPT-2 download helper script from the book's repository
url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch05/"
    "01_main-chapter-code/gpt_download.py"
)
filename = url.split("/")[-1]
urllib.request.urlretrieve(url, filename)
print(f"Downloaded {filename} ✓")

Downloaded gpt_download.py ✓


In [ ]:
from gpt_download import download_and_load_gpt2

# This downloads ~500MB of weights from OpenAI's servers
# Files: checkpoint, encoder.json, hparams.json,
#        model.ckpt.data, model.ckpt.index, model.ckpt.meta, vocab.bpe
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

print("Settings:", settings)
print("Parameter keys:", params.keys())

Settings: {'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12}
Parameter keys: dict_keys(['blocks', 'b', 'g', 'wpe', 'wte'])


In [ ]:
model_configs = {
    "gpt2-small (124M)":  {"emb_dim": 768,  "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)":  {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)":    {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

model_name = "gpt2-small (124M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024})  # GPT-2 was trained with 1024 tokens
NEW_CONFIG.update({"qkv_bias": True})         # GPT-2 uses QKV bias

gpt = GPTModel(NEW_CONFIG)
gpt.eval()
print("Model initialized with random weights — now we override with GPT-2 weights")

Model initialized with random weights — now we override with GPT-2 weights


In [ ]:
import numpy as np

def assign(left, right):
    """Checks shape match then assigns as a trainable Parameter"""
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte"])
    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            params["blocks"][b]["attn"]["c_attn"]["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight   = assign(gpt.trf_blocks[b].att.W_key.weight,   k_w.T)
        gpt.trf_blocks[b].att.W_value.weight  = assign(gpt.trf_blocks[b].att.W_value.weight, v_w.T)
        q_b, k_b, v_b = np.split(
            params["blocks"][b]["attn"]["c_attn"]["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias   = assign(gpt.trf_blocks[b].att.W_key.bias,   k_b)
        gpt.trf_blocks[b].att.W_value.bias  = assign(gpt.trf_blocks[b].att.W_value.bias,  v_b)
        gpt.trf_blocks[b].att.out_proj.weight = assign(gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias   = assign(gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])
        gpt.trf_blocks[b].ff.layers[0].weight = assign(gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias   = assign(gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias   = assign(gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])
        gpt.trf_blocks[b].norm1.scale = assign(gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"])
    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight  = assign(gpt.out_head.weight, params["wte"])  # weight tying

load_weights_into_gpt(gpt, params)
gpt.to(device)
print("GPT-2 pretrained weights loaded successfully ✓")

GPT-2 pretrained weights loaded successfully ✓


<div style="background:linear-gradient(135deg,#4ade8018,#4ade8005);border-left:4px solid #4ade80;border-radius:0 12px 12px 0;padding:20px 28px;margin:32px 0 8px;">
<span style="font-family:monospace;font-size:11px;color:#4ade80;letter-spacing:.12em;text-transform:uppercase;">section 14</span><br>
<strong style="font-size:22px;color:#e8e9f0;">Inference — Text Generation with Pretrained GPT-2</strong><br>
<span style="font-size:13px;color:#9fa3b8;">The moment of truth: coherent text from the model we just built</span>
</div>


In [ ]:
torch.manual_seed(123)

def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None):
    """Extended generation with temperature and top-k sampling"""
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits  = torch.where(
                logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)
        if temperature > 0.0:
            logits   = logits / temperature
            probas   = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probas, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # greedy
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)
print("Output text:")
print(token_ids_to_text(token_ids, tokenizer))

Output text:
Every effort moves you toward finding an ideal new way to practice something that makes you feel alive.


<div style="background:#4ade8012;border:1px solid #4ade8030;border-radius:10px;padding:14px 18px;margin:10px 0;font-size:13px;color:#c8cadc;line-height:1.7;">
<strong style="color:#4ade80;">✅ Note —</strong> This is real, coherent text generated by the same architecture we built from scratch in this notebook. The only difference from our random-weight model: 124 million trained parameters loaded from OpenAI's checkpoint.
</div>


In [ ]:
# Try different prompts
prompts = [
    "The meaning of life is",
    "In the field of machine learning,",
    "Once upon a time there was a",
]

for prompt in prompts:
    torch.manual_seed(42)
    ids = generate(
        model=gpt,
        idx=text_to_token_ids(prompt, tokenizer).to(device),
        max_new_tokens=20,
        context_size=NEW_CONFIG["context_length"],
        top_k=50, temperature=1.0
    )
    print(f"Prompt: {prompt!r}")
    print(f"  → {token_ids_to_text(ids, tokenizer)}")
    print()

Prompt: 'The meaning of life is'
  → The meaning of life is to be in the right place at the right time.

Prompt: 'In the field of machine learning,'
  → In the field of machine learning, the most important thing is to have the right data.

Prompt: 'Once upon a time there was a'
  → Once upon a time there was a man who was very good at his job.



<div style="background:linear-gradient(135deg,#0f1117,#1a1d2e);border:1px solid #2a2d45;border-radius:16px;padding:32px 40px;margin:32px 0 8px;">
<h2 style="color:#e8e9f0;font-weight:300;margin-bottom:16px;">🎯 What we built — from scratch</h2>
<div style="font-size:14px;color:#9fa3b8;line-height:2;">
✅ <strong style="color:#7c8cff;">Self-attention</strong> → dot products, softmax, context vectors<br>
✅ <strong style="color:#2dd4bf;">Trainable W_Q, W_K, W_V</strong> → learned projections<br>
✅ <strong style="color:#f87171;">CausalAttention</strong> → masking, dropout, batch support<br>
✅ <strong style="color:#4ade80;">MultiHeadAttention</strong> → weight-split efficient implementation<br>
✅ <strong style="color:#fbbf24;">LayerNorm</strong> → stabilizes training, mean=0 var=1<br>
✅ <strong style="color:#2dd4bf;">GELU + FeedForward</strong> → 4× expand/contract nonlinearity<br>
✅ <strong style="color:#f472b6;">Shortcut connections</strong> → prevent vanishing gradients<br>
✅ <strong style="color:#7c8cff;">TransformerBlock</strong> → Pre-LayerNorm + MHA + FF + shortcuts<br>
✅ <strong style="color:#4ade80;">GPTModel</strong> → 12 blocks, 124M parameters, 621 MB<br>
✅ <strong style="color:#a78bfa;">Pretrained GPT-2</strong> → loaded OpenAI weights → coherent text<br>
</div>
</div>
